# 03 · Spotify — Segmentación de oyentes (clustering recursivo)

**Problema:** descubrir grupos de oyentes con hábitos parecidos **sin etiquetas previas**, y después explicar qué define a cada grupo.

**Qué vas a ver paso a paso** (la receta que publicó Spotify Engineering: UMAP + HDBSCAN recursivos)

1. Generar 684 oyentes sintéticos con 6 hábitos de escucha.
2. Estandarizar las variables.
3. **UMAP:** proyectar 6 dimensiones a 2 conservando vecindades.
4. **HDBSCAN:** encontrar grupos densos y marcar el ruido.
5. **Recursión:** volver a proyectar y agrupar *dentro* de cada grupo para descubrir subgrupos.
6. **XGBoost + SHAP:** entrenar un modelo sustituto que explique qué hábitos definen cada grupo.

> Datos y parámetros ilustrativos (semilla 42). **Sin datos de usuarios de Spotify ni su configuración interna.** Reproduce `general/es/tools/generate-spotify-demo.py`. La primera ejecución de UMAP tarda unos segundos porque compila código con numba.

In [ ]:
import os
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': .25, 'font.size': 10})
MORADO, ROSA, VERDE, GRIS = '#4D225D', '#E64366', '#2A9D8F', '#9A9A9A'


def cascada(ax, base, nombres, contribs, etiqueta_base='Media', etiqueta_total='Resultado', fmt='{:,.0f}'):
    """Gráfico de cascada: parte de una línea base y suma cada contribución hasta el resultado."""
    xs = [etiqueta_base] + list(nombres) + [etiqueta_total]
    acc, bottoms, heights, colors = base, [0], [base], [GRIS]
    for c in contribs:
        bottoms.append(acc if c >= 0 else acc + c); heights.append(abs(c))
        colors.append(VERDE if c >= 0 else ROSA); acc += c
    bottoms.append(0); heights.append(acc); colors.append(MORADO)
    ax.bar(range(len(xs)), heights, bottom=bottoms, color=colors, width=.65)
    for i, (b, h) in enumerate(zip(bottoms, heights)):
        valor = h if i in (0, len(xs) - 1) else contribs[i - 1]
        texto = fmt.format(valor) if i in (0, len(xs) - 1) else ('+' if valor >= 0 else '') + fmt.format(valor)
        ax.text(i, max(b, b + h), texto, ha='center', va='bottom', fontsize=8)
    ax.axhline(0, color='black', lw=.6)
    ax.set_xticks(range(len(xs)), xs, rotation=30, ha='right')
    return acc

from functools import lru_cache
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, adjusted_rand_score
from sklearn.manifold import trustworthiness
import warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='umap')
import umap, hdbscan
try:
    import xgboost as xgb
except Exception:          # en macOS XGBoost necesita OpenMP: brew install libomp
    xgb = None
    display(Markdown('⚠️ **XGBoost no se ha podido cargar.** En macOS instala OpenMP con `brew install libomp` '
                     'y reinicia el kernel. Los apartados 1–5 funcionan igualmente.'))

SPOTIFY = '#1DB954'
PALETA = [MORADO, ROSA, VERDE, '#F4A261', '#457B9D', '#E9C46A', '#8D5B4C', '#6A994E', '#BC4749', '#3D5A80', '#C77DFF', '#118AB2']
color_de = lambda labels: [PALETA[l % len(PALETA)] if l >= 0 else '#D0D0D0' for l in labels]

## 1 · Generar oyentes sintéticos

Tres perfiles «macro», cada uno con tres variantes, más un 5 % de oyentes aleatorios (ruido):

| Perfil | Minutos/día | Sesiones | Saltos | Descubrimiento | Listas | Noche |
|---|---|---|---|---|---|---|
| Casual de listas | 45 | 3 | 24 % | 22 % | 80 % | 16 % |
| Explorador | 125 | 7 | 45 % | 72 % | 30 % | 34 % |
| Intensivo nocturno | 220 | 11 | 12 % | 18 % | 56 % | 78 % |

Guardamos el perfil real de cada oyente **solo para comprobar** al final; el algoritmo no lo ve.

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)
VARIABLES = ['Minutos', 'Sesiones', 'Saltos', 'Descubrimiento', 'Listas', 'Noche']
PERFILES = ['Casual de listas', 'Explorador', 'Intensivo nocturno']
centres = np.array([[45, 3, .24, .22, .80, .16], [125, 7, .45, .72, .30, .34], [220, 11, .12, .18, .56, .78]])
sd = np.array([8, .55, .035, .045, .045, .045])
chunks, perfil_real, sub_real = [], [], []
for c, centre in enumerate(centres):
    for sub in [-1, 0, 1]:
        offset = np.array([sub*6, sub*.4, sub*.018, sub*.13, -sub*.10, sub*.10])
        chunks.append(rng.normal(centre + offset, sd, (72, 6)))
        perfil_real += [c]*72; sub_real += [c*3 + sub + 1]*72
chunks.append(rng.uniform([15, 1, .02, .02, .02, .02], [280, 14, .72, .98, .98, .98], (36, 6)))
perfil_real += [-1]*36; sub_real += [-1]*36
X = np.vstack(chunks)
X[:, :2] = np.maximum(X[:, :2], .1)
X[:, 2:] = np.clip(X[:, 2:], .001, .999)
perm = rng.permutation(len(X))
X = X[perm].astype(np.float32)
perfil_real, sub_real = np.array(perfil_real)[perm], np.array(sub_real)[perm]

df = pd.DataFrame(X.astype(float), columns=VARIABLES)
df['perfil (oculto)'] = [PERFILES[p] if p >= 0 else 'Ruido' for p in perfil_real]
display(df.head(8).round(2))
print(f'{len(X)} oyentes × {X.shape[1]} variables')

Elige dos variables: en 2D se intuyen grupos, pero cada par de ejes enseña solo una parte. Con 6 dimensiones no podemos mirarlo todo a la vez, y ahí entra UMAP.

In [ ]:
def pares(eje_x='Minutos', eje_y='Noche', colorear_por_perfil_real=False):
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    i, j = VARIABLES.index(eje_x), VARIABLES.index(eje_y)
    c = color_de(perfil_real) if colorear_por_perfil_real else MORADO
    ax.scatter(X[:, i], X[:, j], s=9, alpha=.6, c=c)
    ax.set(xlabel=eje_x, ylabel=eje_y); plt.show()

widgets.interact(pares, eje_x=widgets.Dropdown(options=VARIABLES, value='Minutos'),
                 eje_y=widgets.Dropdown(options=VARIABLES, value='Noche'),
                 colorear_por_perfil_real=widgets.Checkbox(False, description='Mostrar perfil oculto'));

## 2 · Estandarizar

Los minutos van de 15 a 280 y las proporciones de 0 a 1. Sin estandarizar, los minutos dominarían cualquier distancia. Restamos la media y dividimos por la desviación típica.

In [ ]:
scaler = StandardScaler().fit(X)
Z = scaler.transform(X)
pd.DataFrame({'media original': scaler.mean_, 'desv. original': scaler.scale_,
              'media tras escalar': Z.mean(0), 'desv. tras escalar': Z.std(0)}, index=VARIABLES).round(3)

## 3 · UMAP: de 6 dimensiones a 2

UMAP construye un grafo de vecinos en 6D y busca una disposición 2D que lo respete.

- `n_neighbors` pequeño → mira lo **local** (más grupos pequeños); grande → mira lo **global**.
- `min_dist` controla cuánto se apiñan los puntos.

Cambia los parámetros y pulsa **Run Interact** (los resultados se guardan en caché). Colorea por una variable para ver qué gradiente recoge cada zona del mapa.

In [ ]:
@lru_cache(maxsize=None)
def proyectar(n_neighbors, min_dist, seed=SEED, ids=None):
    datos = Z if ids is None else Z[list(ids)]
    return umap.UMAP(n_neighbors=min(n_neighbors, len(datos) - 1), min_dist=min_dist, n_components=2,
                     metric='euclidean', random_state=seed, n_jobs=1, n_epochs=300).fit_transform(datos)

def ver_umap(n_neighbors=45, min_dist=.1, colorear='Noche'):
    emb = proyectar(n_neighbors, min_dist)
    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    if colorear == 'Perfil oculto':
        ax.scatter(*emb.T, s=9, c=color_de(perfil_real))
    else:
        sc = ax.scatter(*emb.T, s=9, c=X[:, VARIABLES.index(colorear)], cmap='viridis'); plt.colorbar(sc, label=colorear)
    ax.set(title=f'UMAP · n_neighbors={n_neighbors}, min_dist={min_dist} · '
                 f'fiabilidad {trustworthiness(Z, emb, n_neighbors=10):.3f}', xticks=[], yticks=[])
    plt.show()

widgets.interact_manual(ver_umap,
    n_neighbors=widgets.SelectionSlider(options=[5, 10, 15, 30, 45, 80, 150], value=45),
    min_dist=widgets.SelectionSlider(options=[0.0, .05, .1, .25, .5, .9], value=.1),
    colorear=widgets.Dropdown(options=VARIABLES + ['Perfil oculto'], value='Noche'));

## 4 · HDBSCAN: grupos por densidad

HDBSCAN busca regiones densas en el mapa 2D. No hay que decirle cuántos grupos hay; los puntos aislados quedan como **ruido** (gris).

- `min_cluster_size`: tamaño mínimo para considerar algo un grupo.
- `min_samples`: cuánto de conservador es al declarar ruido.

Muévelos en tiempo real. El **ARI** compara con el perfil oculto (1 = coincidencia perfecta). El mapa de calor muestra el perfil medio de cada grupo (color = desviaciones respecto a la media; número = valor original).

In [ ]:
emb_raiz = proyectar(45, .1)

def perfil_grupos(ax, ids, labels):
    grupos = sorted(set(labels) - {-1})
    if not grupos: return
    m = np.array([Z[ids][labels == g].mean(0) for g in grupos])
    orig = np.array([X[ids][labels == g].mean(0) for g in grupos])
    ax.imshow(m, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto'); ax.grid(False)
    for (i, j), v in np.ndenumerate(orig):
        ax.text(j, i, f'{v:.0f}' if j < 2 else f'{v:.0%}', ha='center', va='center', fontsize=8)
    ax.set_xticks(range(6), VARIABLES, rotation=30, ha='right')
    ax.set_yticks(range(len(grupos)), [f'C{g+1} (n={int((labels == g).sum())})' for g in grupos])

def agrupar(min_cluster_size=100, min_samples=12):
    labels = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples,
                             cluster_selection_method='eom').fit(emb_raiz).labels_
    k, ruido = len(set(labels) - {-1}), int((labels == -1).sum())
    ok = labels >= 0
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw=dict(width_ratios=[1, 1.1]))
    a1.scatter(*emb_raiz.T, s=9, c=color_de(labels))
    for g in set(labels) - {-1}:
        a1.annotate(f'C{g+1}', emb_raiz[labels == g].mean(0), fontsize=12, weight='bold', ha='center')
    a1.set(title=f'{k} grupos · {ruido} oyentes de ruido', xticks=[], yticks=[])
    perfil_grupos(a2, np.arange(len(X)), labels); a2.set_title('Perfil medio por grupo')
    plt.tight_layout(); plt.show()
    print(f'ARI vs perfil macro: {adjusted_rand_score(perfil_real[ok], labels[ok]):.3f} · '
          f'ARI vs subperfil: {adjusted_rand_score(sub_real[ok], labels[ok]):.3f}  (sin contar ruido)')

widgets.interact(agrupar,
    min_cluster_size=widgets.IntSlider(100, 10, 200, step=5, continuous_update=False, description='min_cluster'),
    min_samples=widgets.IntSlider(12, 1, 30, continuous_update=False, description='min_samples'));

Con `min_cluster_size` grande (100–150) aparecen los 3 perfiles macro; con valores pequeños (~45) aparecen directamente subgrupos. En lugar de elegir un único nivel, Spotify **recursa**: agrupa a lo grande y vuelve a agrupar dentro.

## 5 · Recursión: subgrupos dentro de cada grupo

Fijamos la raíz (`min_cluster_size=100`, `min_samples=12`). Para cada grupo con ≥ 60 oyentes se calcula un **UMAP nuevo** solo con sus miembros (mismas 6 variables, `n_neighbors=15`) y un HDBSCAN más fino (`18`, `7`).

In [ ]:
labels_raiz = hdbscan.HDBSCAN(min_cluster_size=100, min_samples=12, cluster_selection_method='eom').fit(emb_raiz).labels_
hijos = {}
for g in sorted(set(labels_raiz) - {-1}):
    ids = np.where(labels_raiz == g)[0]
    if len(ids) < 60: continue
    emb = proyectar(15, .1, SEED + int(g) + 1, tuple(ids))
    hijos[g] = (ids, emb, hdbscan.HDBSCAN(min_cluster_size=18, min_samples=7, cluster_selection_method='eom').fit(emb).labels_)
    print(f'C{g+1}: {len(ids)} oyentes → {len(set(hijos[g][2]) - {-1})} subgrupos')

def ver_hijo(grupo):
    ids, emb, lab = hijos[grupo]
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw=dict(width_ratios=[.8, 1, 1.1]))
    axs[0].scatter(*emb_raiz.T, s=6, c=['#D0D0D0']*len(X))
    axs[0].scatter(*emb_raiz[ids].T, s=8, c=PALETA[grupo % len(PALETA)])
    axs[0].set(title=f'C{grupo+1} en el mapa raíz', xticks=[], yticks=[])
    axs[1].scatter(*emb.T, s=12, c=color_de(lab))
    axs[1].set(title=f'UMAP propio de C{grupo+1} · {len(set(lab) - {-1})} subgrupos', xticks=[], yticks=[])
    perfil_grupos(axs[2], ids, lab); axs[2].set_title('Perfil de los subgrupos')
    plt.tight_layout(); plt.show()
    ok = lab >= 0
    print(f'ARI de los subgrupos vs subperfil oculto: {adjusted_rand_score(sub_real[ids][ok], lab[ok]):.3f}')

widgets.interact(ver_hijo, grupo=widgets.ToggleButtons(options=[(f'C{g+1}', g) for g in hijos], description='Grupo'));

## 6 · Explicar los grupos con XGBoost + SHAP

Los grupos salen de un mapa 2D, que no se interpreta directamente. Para explicarlos entrenamos, por cada grupo, un clasificador **«¿pertenece a este grupo?»** sobre las 6 variables originales. Después, TreeSHAP exacto reparte cada predicción entre las variables (en log-odds).

- F1 y AUC miden la **fidelidad del sustituto** frente a las etiquetas descubiertas, no la calidad de los grupos.

In [ ]:
assert xgb is not None, 'XGBoost no está disponible: en macOS ejecuta `brew install libomp` y reinicia el kernel.'
modelos, todos = {}, np.arange(len(X))
for g in sorted(set(labels_raiz) - {-1}):
    yg = (labels_raiz == g).astype(int)
    tr, te = train_test_split(todos, test_size=.25, stratify=yg, random_state=SEED)
    m = xgb.XGBClassifier(n_estimators=90, max_depth=3, learning_rate=.08, objective='binary:logistic',
                          eval_metric='logloss', tree_method='hist', n_jobs=1, random_state=SEED).fit(X[tr], yg[tr])
    contrib = m.get_booster().predict(xgb.DMatrix(X), pred_contribs=True)
    prob = m.predict_proba(X)[:, 1]
    modelos[g] = dict(shap=contrib[:, :-1], base=float(contrib[0, -1]), prob=prob,
                      f1=f1_score(yg[te], prob[te] > .5), auc=roc_auc_score(yg[te], prob[te]))
pd.DataFrame({f'C{g+1}': dict(F1=v['f1'], AUC=v['auc'], miembros=int((labels_raiz == g).sum())) for g, v in modelos.items()}).T.round(3)

In [ ]:
def explicar(grupo, oyente=0):
    v = modelos[grupo]
    miembros = np.where(labels_raiz == grupo)[0]
    i = miembros[min(oyente, len(miembros) - 1)]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    imp = np.abs(v['shap'][labels_raiz == grupo]).mean(0)
    orden = np.argsort(imp)
    a1.barh(np.array(VARIABLES)[orden], imp[orden], color=PALETA[grupo % len(PALETA)])
    a1.set(title=f'¿Qué define a C{grupo+1}? (|SHAP| medio en sus miembros)', xlabel='log-odds')
    cascada(a2, v['base'], VARIABLES, v['shap'][i], 'Base', 'Oyente', '{:.2f}')
    a2.set(ylabel='log-odds de pertenecer', title=f'Oyente #{i} · p(C{grupo+1}) = {v["prob"][i]:.1%}')
    plt.tight_layout(); plt.show()
    display(df.iloc[[i], :6].round(2))

g0 = next(iter(modelos))
sel_g = widgets.ToggleButtons(options=[(f'C{g+1}', g) for g in modelos], description='Grupo')
sel_o = widgets.IntSlider(0, 0, 200, description='Oyente nº')
display(sel_g, sel_o, widgets.interactive_output(explicar, {'grupo': sel_g, 'oyente': sel_o}))

## Para llevarse

- **UMAP** hace visible la estructura; **HDBSCAN** la convierte en grupos sin fijar k y aparta el ruido.
- La **recursión** permite ver el bosque y los árboles: perfiles amplios y matices dentro de cada uno.
- Un **modelo sustituto + SHAP** traduce grupos abstractos a lenguaje de negocio: «escuchan de noche y muchas sesiones».